## Step 1: Environment Setup

Installing the `google-cloud-aiplatform` SDK, which provides the Python client libraries needed to interact with Vertex AI and Google Cloud Storage from this Workbench notebook instance.

!pip3 install --upgrade --quiet google-cloud-aiplatform

## Step 2: Create / Verify Google Cloud Storage Bucket

Connecting to GCS using the `google-cloud-storage` client. This cell checks whether a bucket named `23f1000932-mlops-week1` already exists in the project `project-bcb80534-6ef1-4dcc-951`. If it doesn't exist, it creates a new bucket in the `us-central1` region. This bucket will be used to store the training/evaluation data and all model artifacts produced by the pipeline.

**Note:** The Vertex AI Workbench instance runs under the default Compute Engine service account. This service account initially lacked Storage permissions, which caused a `403 Forbidden` error on the first attempt. This was resolved by granting the **Storage Admin** role to the service account `17077469791-compute@developer.gserviceaccount.com` via IAM & Admin.

In [29]:
from google.cloud import storage

PROJECT_ID = "project-352c5688-c8ba-4f4e-aaa"
BUCKET_NAME = "24f2000612-mlops-week1"
LOCATION = "us-central1"

client = storage.Client(project=PROJECT_ID)

# Create bucket if it doesn't exist
bucket = client.lookup_bucket(BUCKET_NAME)
if bucket is None:
    bucket = client.create_bucket(BUCKET_NAME, location=LOCATION)
    print("Bucket created:", bucket.name)
else:
    print("Bucket already exists:", bucket.name)

Bucket already exists: 24f2000612-mlops-week1


## Step 3: Prepare and Upload IRIS Dataset to GCS

Loading the local `iris.csv` dataset and splitting it into a training set (80%) and an evaluation set (20%), stratified by the `species` column to preserve class balance. Both splits are saved locally as CSV files and then uploaded to the GCS bucket under the `data/` prefix:
- `gs://23f1000932-mlops-week1/data/train.csv`
- `gs://23f1000932-mlops-week1/data/eval.csv`

This separates the data used for training from the data used later for evaluation/inference, and stores both in cloud storage rather than relying on local notebook disk.

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("iris.csv")

train, eval_set = train_test_split(
    data, test_size=0.2, stratify=data["species"], random_state=42
)

train.to_csv("train.csv", index=False)
eval_set.to_csv("eval.csv", index=False)

bucket = client.bucket(BUCKET_NAME)
bucket.blob("data/train.csv").upload_from_filename("train.csv")
bucket.blob("data/eval.csv").upload_from_filename("eval.csv")

print("Uploaded train.csv and eval.csv to GCS")

Uploaded train.csv and eval.csv to GCS


## Step 4: Verify Data Upload

Listing all objects under the `data/` prefix in the GCS bucket to confirm that `train.csv` and `eval.csv` were successfully uploaded.

In [31]:
blobs = list(client.list_blobs(BUCKET_NAME, prefix="data/"))
for b in blobs:
    print(b.name)

data/eval.csv
data/train.csv


In [ ]:
## Step 5: Train the IRIS Classification Model (Run 1)

This cell implements the full training pipeline as a self-contained block:
1. Fetches `train.csv` from the GCS bucket (not from local disk), confirming the training pipeline reads its input data from cloud storage.
2. Trains a `DecisionTreeClassifier` (max_depth=3) on the four IRIS features (sepal length/width, petal length/width) to predict species.
3. Generates a UTC timestamp representing this specific training execution.
4. Saves the trained model (`model.joblib`) and a training log (`train_log.txt`) locally under a folder named after the timestamp.
5. Uploads both artifacts to GCS under `artifacts/<timestamp>/`, so each training run is organized and traceable by its execution time.

This is **Run 1** of the training pipeline. Run timestamp: `2026-06-21T08-17-41`.

In [37]:
import os
import joblib
import pandas as pd
from datetime import datetime
from sklearn.tree import DecisionTreeClassifier
from google.cloud import storage

PROJECT_ID = "project-352c5688-c8ba-4f4e-aaa"
BUCKET_NAME = "24f2000612-mlops-week1"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

# 1. Fetch training data from GCS
bucket.blob("data/train.csv").download_to_filename("train.csv")
train = pd.read_csv("train.csv")

X_train = train[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y_train = train["species"]

# 2. Train model
model = DecisionTreeClassifier(max_depth=3, random_state=1)
model.fit(X_train, y_train)

# 3. Create timestamped run folder
timestamp = datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%S")
local_dir = f"artifacts/{timestamp}"
os.makedirs(local_dir, exist_ok=True)

model_path = f"{local_dir}/model.joblib"
joblib.dump(model, model_path)

log_path = f"{local_dir}/train_log.txt"
with open(log_path, "w") as f:
    f.write(f"Training run: {timestamp}\n")
    f.write(f"Model: DecisionTreeClassifier(max_depth=3)\n")
    f.write(f"Train samples: {len(train)}\n")

# 4. Upload artifacts to GCS under timestamped prefix
bucket.blob(f"artifacts/{timestamp}/model.joblib").upload_from_filename(model_path)
bucket.blob(f"artifacts/{timestamp}/train_log.txt").upload_from_filename(log_path)

print(f"Training complete. Artifacts uploaded to gs://{BUCKET_NAME}/artifacts/{timestamp}/")
print("RUN_TIMESTAMP:", timestamp)

/var/tmp/ipykernel_14538/4083146081.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%S")


Training complete. Artifacts uploaded to gs://24f2000612-mlops-week1/artifacts/2026-06-21T08-39-23/
RUN_TIMESTAMP: 2026-06-21T08-39-23


In [ ]:
## Step 6: Verify Artifacts for Run 1

Listing all objects under the `artifacts/` prefix to confirm the model and training log for the first run were uploaded correctly to `artifacts/2026-06-21T08-17-41/`.

In [34]:
blobs = list(client.list_blobs(BUCKET_NAME, prefix="artifacts/"))
for b in blobs:
    print(b.name)

artifacts/2026-06-21T08-36-58/model.joblib
artifacts/2026-06-21T08-36-58/train_log.txt


In [ ]:
## Step 7: Run Inference on Evaluation Set (Run 1)

This is a logically separate inference step that:
1. Downloads the trained model **back from GCS** (using the Run 1 timestamp) rather than reusing the in-memory model object — this proves the model can be retrieved independently of the training session.
2. Downloads `eval.csv` from GCS.
3. Runs predictions on the evaluation set and computes accuracy.
4. Prints sample predictions vs actual labels.
5. Writes an inference log (`inference_log.txt`) and uploads it back to the same `artifacts/<timestamp>/` folder, keeping training and inference results for a given run together.

Result for Run 1: **Eval accuracy = 0.967**

In [36]:
from sklearn import metrics

RUN_TIMESTAMP = "2026-06-21T08-36-58"  # paste your timestamp here

# 1. Fetch model from GCS using the run timestamp
bucket.blob(f"artifacts/{RUN_TIMESTAMP}/model.joblib").download_to_filename("model_loaded.joblib")
model_loaded = joblib.load("model_loaded.joblib")

# 2. Fetch eval data from GCS
bucket.blob("data/eval.csv").download_to_filename("eval.csv")
eval_data = pd.read_csv("eval.csv")

X_eval = eval_data[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y_eval = eval_data["species"]

# 3. Run inference
predictions = model_loaded.predict(X_eval)
accuracy = metrics.accuracy_score(y_eval, predictions)

print(f"Inference run on model from {RUN_TIMESTAMP}")
print(f"Eval accuracy: {accuracy:.3f}")

for actual, pred in zip(y_eval.head(10), predictions[:10]):
    print(f"Actual: {actual} | Predicted: {pred}")

# 4. Upload inference log back to GCS
log_text = f"Inference run for model {RUN_TIMESTAMP}\nAccuracy: {accuracy:.3f}\n"
with open("inference_log.txt", "w") as f:
    f.write(log_text)
bucket.blob(f"artifacts/{RUN_TIMESTAMP}/inference_log.txt").upload_from_filename("inference_log.txt")

print("Inference log uploaded to GCS")

Inference run on model from 2026-06-21T08-36-58
Eval accuracy: 0.967
Actual: setosa | Predicted: setosa
Actual: virginica | Predicted: virginica
Actual: versicolor | Predicted: versicolor
Actual: versicolor | Predicted: versicolor
Actual: setosa | Predicted: setosa
Actual: versicolor | Predicted: versicolor
Actual: setosa | Predicted: setosa
Actual: setosa | Predicted: setosa
Actual: virginica | Predicted: virginica
Actual: versicolor | Predicted: versicolor
Inference log uploaded to GCS


In [ ]:
## Step 8: Train the IRIS Classification Model (Run 2)

Re-executing the same training pipeline as Step 5 a second time. Because the timestamp is generated fresh on each execution, this produces a **new, separate artifacts folder** in GCS, distinct from Run 1 — demonstrating that multiple pipeline executions can be tracked independently without overwriting previous results.

Run timestamp: `2026-06-21T08-22-00`.

In [38]:
blobs = list(client.list_blobs(BUCKET_NAME, prefix="artifacts/"))
for b in blobs:
    print(b.name)

artifacts/2026-06-21T08-36-58/inference_log.txt
artifacts/2026-06-21T08-36-58/model.joblib
artifacts/2026-06-21T08-36-58/train_log.txt
artifacts/2026-06-21T08-39-23/model.joblib
artifacts/2026-06-21T08-39-23/train_log.txt


In [40]:
RUN_TIMESTAMP = "2026-06-21T08-36-58"  # second run

bucket.blob(f"artifacts/{RUN_TIMESTAMP}/model.joblib").download_to_filename("model_loaded.joblib")
model_loaded = joblib.load("model_loaded.joblib")

bucket.blob("data/eval.csv").download_to_filename("eval.csv")
eval_data = pd.read_csv("eval.csv")

X_eval = eval_data[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y_eval = eval_data["species"]

predictions = model_loaded.predict(X_eval)
accuracy = metrics.accuracy_score(y_eval, predictions)

print(f"Inference run on model from {RUN_TIMESTAMP}")
print(f"Eval accuracy: {accuracy:.3f}")

for actual, pred in zip(y_eval.head(10), predictions[:10]):
    print(f"Actual: {actual} | Predicted: {pred}")

log_text = f"Inference run for model {RUN_TIMESTAMP}\nAccuracy: {accuracy:.3f}\n"
with open("inference_log.txt", "w") as f:
    f.write(log_text)
bucket.blob(f"artifacts/{RUN_TIMESTAMP}/inference_log.txt").upload_from_filename(log_text and "inference_log.txt")

print("Inference log uploaded to GCS")

Inference run on model from 2026-06-21T08-36-58
Eval accuracy: 0.967
Actual: setosa | Predicted: setosa
Actual: virginica | Predicted: virginica
Actual: versicolor | Predicted: versicolor
Actual: versicolor | Predicted: versicolor
Actual: setosa | Predicted: setosa
Actual: versicolor | Predicted: versicolor
Actual: setosa | Predicted: setosa
Actual: setosa | Predicted: setosa
Actual: virginica | Predicted: virginica
Actual: versicolor | Predicted: versicolor
Inference log uploaded to GCS


In [42]:
bucket.blob(f"artifacts/{RUN_TIMESTAMP}/inference_log.txt").upload_from_filename("inference_log.txt")

In [43]:
lobs = list(client.list_blobs(BUCKET_NAME, prefix="artifacts/"))
for b in blobs:
    print(b.name)

artifacts/2026-06-21T08-36-58/inference_log.txt
artifacts/2026-06-21T08-36-58/model.joblib
artifacts/2026-06-21T08-36-58/train_log.txt
artifacts/2026-06-21T08-39-23/model.joblib
artifacts/2026-06-21T08-39-23/train_log.txt
